In [0]:
# Databricks notebook source
# WA Road Incidents - Structured Streaming
# Reads from Event Hubs, parses JSON, computes hourly hotspot counts,
# writes both raw incidents and hotspot aggregates to Delta tables.
# Designed to run as a scheduled Databricks Job (every 5 minutes),
# using trigger(availableNow=True) so each run processes what's
# currently available then exits cleanly - required on serverless compute.

# COMMAND ----------

EVENTHUB_NAMESPACE = "PASTE_YOUR_NAMESPACE_STRING_HERE"
EVENTHUB_CONNECTION_STRING = "PASTE_YOUR_CONNECTION_STRING_HERE"

kafka_options = {
    "kafka.bootstrap.servers": f"{EVENTHUB_NAMESPACE}:9093",
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.sasl.jaas.config": f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="$ConnectionString" password="{EVENTHUB_CONNECTION_STRING}";',
    "subscribe": "wa-road-incidents",
    "startingOffsets": "earliest"
}

raw_stream = spark.readStream.format("kafka").options(**kafka_options).load()

# COMMAND ----------

from pyspark.sql.types import StructType, StringType, IntegerType, DoubleType
from pyspark.sql.functions import col, from_json, to_timestamp, window, count

incident_schema = StructType() \
    .add("attributes", StructType()
        .add("FID", IntegerType())
        .add("Id", IntegerType())
        .add("Location", StringType())
        .add("IncidentTy", StringType())
        .add("ClosureTyp", StringType())
        .add("TrafficCon", StringType())
        .add("EntryDate", StringType())
        .add("UpdateDate", StringType())
        .add("Road", StringType())
        .add("Region", StringType())
        .add("Suburb", StringType())
        .add("GlobalID", StringType())
        .add("TrafficImp", StringType())
        .add("SeeMoreUrl", StringType())
        .add("LocalRoadName", StringType())
    ) \
    .add("geometry", StructType()
        .add("x", DoubleType())
        .add("y", DoubleType())
    )

parsed_stream = raw_stream.select(
    from_json(col("value").cast("string"), incident_schema).alias("data")
).select("data.attributes.*", "data.geometry.x", "data.geometry.y")

# COMMAND ----------

parsed_with_time = parsed_stream.withColumn(
    "event_time",
    to_timestamp(col("UpdateDate"), "dd/MM/yyyy HH:mm:ss")
)

# COMMAND ----------

hotspot_counts = parsed_with_time \
    .withWatermark("event_time", "10 minutes") \
    .groupBy(
        window(col("event_time"), "1 hour"),
        col("Road")
    ) \
    .agg(count("*").alias("incident_count"))

# COMMAND ----------

# Write raw parsed incidents - one row per incident, append-only
raw_query = parsed_with_time.writeStream \
    .format("delta") \
    .option("checkpointLocation", "/Volumes/ingestion/default/checkpoints/wa_raw_incidents") \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .toTable("ingestion.default.raw_incidents")

raw_query.awaitTermination()
print("raw_incidents write complete")

# COMMAND ----------

# Write hourly hotspot counts per road - full aggregation state rewritten each run
hotspot_query = hotspot_counts.writeStream \
    .format("delta") \
    .option("checkpointLocation", "/Volumes/ingestion/default/checkpoints/wa_hotspots_table_v2") \
    .outputMode("complete") \
    .trigger(availableNow=True) \
    .toTable("ingestion.default.road_hotspots")

hotspot_query.awaitTermination()
print("road_hotspots write complete")

raw_incidents write complete
road_hotspots write complete
